### Structured output

Models can be requested to provide their response in a formt matching in a given schema.

### Pydantic

Pydantic models provide the richest feature set with field validation.

In [7]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001AF8580EE40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001AF8580FB60>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The name of the director for this movie")
    rating:float=Field(description="The movies ratings out of 10")

In [18]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)


In [19]:
response = model_with_structure.invoke("Provide details of the Movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me see what I need to do. The available tool is the Movie function, which requires title, year, director, and rating. I need to provide those parameters for Inception. First, I should recall the movie's details. Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Wait, but the rating parameter is a number out of 10, so I'll use that. Let me make sure all required fields are included: title, year, director, rating. Yep, that's covered. So I'll structure the tool call with those parameters.\n", 'tool_calls': [{'id': 'yv7qty188', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 196, 'prompt_tokens': 228, 'total_tokens': 424, 'completion_time': 

### Nested Structure



In [21]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    generes: list[str]
    budget: float | None = Field (None, description="Budget in millions USD")
    
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details of the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], generes=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

### TypeDict

This is simple python alternative built-in type ideal for run time validation

In [22]:
from typing import TypedDict, Annotated
class MovieDict(TypedDict):
    title: Annotated[str, ..., "The title of the Movie"]
    year: Annotated[str,...,"The year of the movie released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[str, ..., "The movie's rating out of 10"]

model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon',
 'rating': '8.0',
 'title': 'The Avengers',
 'year': '2012'}

In [24]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    generes: list[str]
    budget: float | None = Field (None, description="Budget in millions USD")
    
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details of the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'generes': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'The Avengers',
 'year': 2012}

In [25]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses

A data class is class contiaing data there are really any restrictsion. You create it using decorator.

In [26]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain_protocol.protocol import ResultData
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """ Contact information of a person """
    name: str = Field(description = "The name of the person")
    email: str = Field(description = "The email address of the person")
    phone: str = Field(description = "The phone number of the person")


agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo 
)

result = agent.invoke(
    {
        "messages":
        [
            {
                "role":"user",
                "content":"Extract contact infor form: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [33]:
from langchain_protocol.protocol import ResultData
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """ Contact information of a person """
    name: str = Field(description = "The name of the person")
    email: str = Field(description = "The email address of the person")
    phone: str = Field(description = "The phone number of the person")


agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo 
)

result = agent.invoke(
    {
        "messages":
        [
            {
                "role":"user",
                "content":"Extract contact infor form: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [ ]:
### DataClass

from dataclasses import dataclass
from langchain.agents import dataclass

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

agent = create_agent (
    model = "gpt-5",
    response_format=ContactInfo
)

result =  agent.invoke(
    {
        "messages": 
        [
            {
                "role":"user",
                "content":"Extract contact infor from: John Doe, john@example.com, (333) 123-5555"
            }
        ]
    }
)
result["structured_response"]

SyntaxError: invalid syntax (1066330547.py, line 21)